# SpaceX Launch Analysis with SQL

This notebook explores the historical SpaceX launch dataset using SQL. It is designed to run locally in Jupyter or Google Colab without a cloud database account, credentials, or SQL magic extensions.

**Tools:** Python, pandas, SQLite, SQL

The notebook uses the repository's `Spacex.csv` file and creates a temporary SQLite database in memory. The source CSV is not modified.

## 1. Load the data and create a local SQL table

Make sure `Spacex.csv` is in the same folder as this notebook. If running in Colab, upload the CSV or clone/download the repository first.

In [10]:

!wget -q -O Spacex.csv \
  https://raw.githubusercontent.com/olafisoyetope44-ship-it/Applied-Data-Capstone/main/Spacex.csv

In [11]:
from pathlib import Path
import sqlite3
import pandas as pd

csv_path = Path("Spacex.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "Spacex.csv was not found. Place it beside this notebook, "
        "or update csv_path to the file's location."
    )

df = pd.read_csv(csv_path)
df.columns = [
    "Date", "TimeUTC", "BoosterVersion", "LaunchSite", "Payload",
    "PayloadMassKG", "Orbit", "Customer", "MissionOutcome", "LandingOutcome"
]
df["PayloadMassKG"] = pd.to_numeric(df["PayloadMassKG"], errors="coerce")

# In-memory SQLite database: no server, password, or external service required.
conn = sqlite3.connect(":memory:")
df.to_sql("SpaceX", conn, if_exists="replace", index=False)

print(f"Loaded {len(df)} launch records into SQLite table SpaceX.")
display(df.head())

Loaded 101 launch records into SQLite table SpaceX.


,Date,TimeUTC,BoosterVersion,LaunchSite,Payload,PayloadMassKG,Orbit,Customer,MissionOutcome,LandingOutcome
0,2010-04-06,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-08-12,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-08-10,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-01-03,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## 2. Explore the table


In [12]:
pd.read_sql_query("SELECT * FROM SpaceX LIMIT 10;", conn)

,Date,TimeUTC,BoosterVersion,LaunchSite,Payload,PayloadMassKG,Orbit,Customer,MissionOutcome,LandingOutcome
0,2010-04-06,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-08-12,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-08-10,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-01-03,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt
5,2013-09-23,16:00:00,F9 v1.1 B1003,VAFB SLC-4E,CASSIOPE,500,Polar LEO,MDA,Success,Uncontrolled (ocean)
6,2013-03-12,22:41:00,F9 v1.1,CCAFS LC-40,SES-8,3170,GTO,SES,Success,No attempt
7,2014-06-01,22:06:00,F9 v1.1,CCAFS LC-40,Thaicom 6,3325,GTO,Thaicom,Success,No attempt
8,2014-04-18,19:25:00,F9 v1.1,CCAFS LC-40,SpaceX CRS-3,2296,LEO (ISS),NASA (CRS),Success,Controlled (ocean)
9,2014-07-14,15:15:00,F9 v1.1,CCAFS LC-40,OG2 Mission 1 6 Orbcomm-OG2 satellites,1316,LEO,Orbcomm,Success,Controlled (ocean)


## 3. SQL analysis tasks

Each section states the business question, runs a SQL query, and displays the result as a pandas DataFrame.

### Task 1 — What launch sites appear in the dataset?


In [13]:
query = """
SELECT DISTINCT LaunchSite
FROM SpaceX
ORDER BY LaunchSite;
"""
pd.read_sql_query(query, conn)

,LaunchSite
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E


### Task 2 — Which launches took place at sites beginning with `CCA`?


In [14]:
query = """
SELECT Date, BoosterVersion, LaunchSite, Payload, PayloadMassKG, Orbit
FROM SpaceX
WHERE LaunchSite LIKE 'CCA%'
ORDER BY Date
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,Date,BoosterVersion,LaunchSite,Payload,PayloadMassKG,Orbit
0,2010-04-06,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO
1,2010-08-12,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS)
2,2012-05-22,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS)
3,2012-08-10,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS)
4,2013-01-03,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS)
5,2013-03-12,F9 v1.1,CCAFS LC-40,SES-8,3170,GTO
6,2014-04-18,F9 v1.1,CCAFS LC-40,SpaceX CRS-3,2296,LEO (ISS)
7,2014-05-08,F9 v1.1,CCAFS LC-40,AsiaSat 8,4535,GTO
8,2014-06-01,F9 v1.1,CCAFS LC-40,Thaicom 6,3325,GTO
9,2014-07-09,F9 v1.1 B1011,CCAFS LC-40,AsiaSat 6,4428,GTO


### Task 3 — What is the total payload mass for NASA CRS missions?


In [15]:
query = """
SELECT SUM(PayloadMassKG) AS TotalPayloadMassKG
FROM SpaceX
WHERE Customer LIKE '%NASA (CRS)%';
"""
pd.read_sql_query(query, conn)

,TotalPayloadMassKG
0,48213


### Task 4 — What is the average payload mass for the F9 v1.1 booster version?


In [16]:
query = """
SELECT ROUND(AVG(PayloadMassKG), 2) AS AveragePayloadMassKG
FROM SpaceX
WHERE BoosterVersion LIKE 'F9 v1.1%';
"""
pd.read_sql_query(query, conn)

,AveragePayloadMassKG
0,2534.67


### Task 5 — What is the earliest recorded successful ground-pad landing?


In [17]:
query = """
SELECT MIN(Date) AS FirstSuccessfulGroundPadLandingDate
FROM SpaceX
WHERE LandingOutcome LIKE '%Success%ground pad%';
"""
pd.read_sql_query(query, conn)

,FirstSuccessfulGroundPadLandingDate
0,2015-12-22


### Task 6 — Which booster versions had successful drone-ship landings with payloads between 4,000 and 6,000 kg?


In [18]:
query = """
SELECT BoosterVersion, PayloadMassKG, LandingOutcome
FROM SpaceX
WHERE LandingOutcome LIKE '%Success%drone ship%'
  AND PayloadMassKG BETWEEN 4000 AND 6000
ORDER BY PayloadMassKG;
"""
pd.read_sql_query(query, conn)

,BoosterVersion,PayloadMassKG,LandingOutcome
0,F9 FT B1026,4600,Success (drone ship)
1,F9 FT B1022,4696,Success (drone ship)
2,F9 FT B1031.2,5200,Success (drone ship)
3,F9 FT B1021.2,5300,Success (drone ship)


### Task 7 — How many missions succeeded or failed?


In [19]:
query = """
SELECT MissionOutcome, COUNT(*) AS MissionCount
FROM SpaceX
GROUP BY MissionOutcome
ORDER BY MissionCount DESC;
"""
pd.read_sql_query(query, conn)

,MissionOutcome,MissionCount
0,Success,98
1,Success (payload status unclear),1
2,Success,1
3,Failure (in flight),1


### Task 8 — Which booster versions carried the maximum payload mass?


In [20]:
query = """
SELECT BoosterVersion, PayloadMassKG
FROM SpaceX
WHERE PayloadMassKG = (SELECT MAX(PayloadMassKG) FROM SpaceX)
ORDER BY BoosterVersion;
"""
pd.read_sql_query(query, conn)

,BoosterVersion,PayloadMassKG
0,F9 B5 B1048.4,15600
1,F9 B5 B1048.5,15600
2,F9 B5 B1049.4,15600
3,F9 B5 B1049.5,15600
4,F9 B5 B1049.7,15600
5,F9 B5 B1051.3,15600
6,F9 B5 B1051.4,15600
7,F9 B5 B1051.6,15600
8,F9 B5 B1056.4,15600
9,F9 B5 B1058.3,15600


### Task 9 — Which launches in 2015 had a drone-ship landing failure?


In [21]:
query = """
SELECT Date, BoosterVersion, LaunchSite, LandingOutcome
FROM SpaceX
WHERE LandingOutcome LIKE '%Failure%drone ship%'
  AND Date BETWEEN '2015-01-01' AND '2015-12-31'
ORDER BY Date;
"""
pd.read_sql_query(query, conn)

,Date,BoosterVersion,LaunchSite,LandingOutcome
0,2015-04-14,F9 v1.1 B1015,CCAFS LC-40,Failure (drone ship)
1,2015-10-01,F9 v1.1 B1012,CCAFS LC-40,Failure (drone ship)


### Task 10 — How were landing outcomes distributed during the selected study period?


In [22]:
query = """
SELECT LandingOutcome, COUNT(*) AS OutcomeCount
FROM SpaceX
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY LandingOutcome
ORDER BY OutcomeCount DESC;
"""
pd.read_sql_query(query, conn)

,LandingOutcome,OutcomeCount
0,No attempt,10
1,Success (drone ship),6
2,Success (ground pad),5
3,Failure (drone ship),5
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Precluded (drone ship),1
7,Failure (parachute),1


## 4. Data quality checks

Check missing values and date coverage before drawing conclusions. The launch records are historical, and findings describe this dataset rather than guaranteeing future landing outcomes.

In [23]:
quality_checks = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(1)
})
display(quality_checks)

pd.read_sql_query(
    "SELECT MIN(Date) AS FirstLaunchDate, MAX(Date) AS LastLaunchDate, COUNT(*) AS RecordCount FROM SpaceX;",
    conn
)

,missing_values,missing_percent
Date,0,0.0
TimeUTC,0,0.0
BoosterVersion,0,0.0
LaunchSite,0,0.0
Payload,0,0.0
PayloadMassKG,0,0.0
Orbit,0,0.0
Customer,0,0.0
MissionOutcome,0,0.0
LandingOutcome,0,0.0


,FirstLaunchDate,LastLaunchDate,RecordCount
0,2010-04-06,2020-11-25,101


## Summary

This notebook demonstrates SQL filtering, pattern matching, aggregation, grouping, date filtering, and subqueries using a local SQLite database. It requires only Python, pandas, and the `Spacex.csv` file.

To rerun the analysis, restart the kernel and run the notebook from top to bottom. Close the in-memory database when finished.

In [24]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.
